# Modelica to PyPowSyBl: Dynamic Simulation Toolkit

### Industrialized Workflow
1. **Create Network:** Programmatically build the static loadflow structure via PyPowSyBl.
2. **Initialize Loadflow:** Define load and generation profiles.
3. **Run Loadflow:** Execute the base AC Loadflow using OpenLF.
4. **Dynamic Models Setup:** Map precompiled models (e.g., `GeneratorPV`) directly from the Dynawo database.
5. **Physical & Initialization Parameters:** Build the `.PAR` files using physical parameters and Loadflow initialization references.
6. **Execute Dynamic Simulation:** Run Dynawo time-domain simulation for events.

## Step 0: Environment Setup (Dynawo Install & Config)

Before running dynamic simulations, we need the Dynawo engine installed and configured so PyPowSyBl can find it.

In [ ]:
# --- 1. ENVIRONMENT SETUP: CHECK DYNAWO INSTALLATION ---
import os
import re
import datetime


def verify_dynawo_installation():
    """Finds and validates the Dynawo installation path from the config file."""
    config_directory = os.path.expanduser("~/.itools")
    config_filepath = os.path.join(config_directory, "config.yml")
    dynawo_path = None

    # Check if config.yml exists and extract the homeDir value
    if os.path.exists(config_filepath):
        with open(config_filepath, "r") as file:
            content = file.read()
            match = re.search(r'homeDir:\s*["\']?([^"\'\n]+)["\']?', content)
            if match:
                dynawo_path = match.group(1)

    # Verify if a path was extracted and if it actually exists
    if not dynawo_path or not os.path.exists(dynawo_path):
        raise FileNotFoundError(
            f"Error! No valid Dynawo installation found on the system. "
            f"Detected path: '{dynawo_path}'. Make sure Dynawo is installed and properly configured in {config_filepath}."
        )

    print(f"Configuration correct! Dynawo installation validated at: {dynawo_path}")

    # Save the verified path in an environment variable for subsequent steps
    os.environ["DYNAWO_SYS_PATH"] = dynawo_path
    return dynawo_path, config_directory


# Execute verification
dynawo_sys_path, global_config_dir = verify_dynawo_installation()

In [ ]:
# --- 2. ENVIRONMENT SETUP: CONFIGURE TEMPLATES ---
def configure_environment_template(dynawo_path, config_directory):
    """Overwrites the PyPowSyBl configuration with the local project template."""
    target_config_filepath = os.path.join(config_directory, "config.yml")
    template_config_filepath = "./getting_started_data/config.yml"
    current_working_directory = os.getcwd()

    if not dynawo_path:
        raise ValueError("The Dynawo path is not defined. Please run the previous cell first.")

    # Create a backup of the existing configuration if it exists
    if os.path.exists(target_config_filepath):
        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        backup_filepath = os.path.join(config_directory, f"config_{timestamp}.yml")
        os.rename(target_config_filepath, backup_filepath)
        print(f"Backup created at: {backup_filepath}")

    with open(template_config_filepath, "r") as file:
        template_content = file.read()

    # Inject dynamic paths into the template
    template_content = template_content.replace('"WORKING_DIR/dynawo"', f'"{dynawo_path}"')
    template_content = template_content.replace("WORKING_DIR", current_working_directory)

    os.makedirs(config_directory, exist_ok=True)
    with open(target_config_filepath, "w") as file:
        file.write(template_content)

    print(f"\nDone! Template successfully configured and saved to {target_config_filepath}:\n")
    print(template_content)


# Execute configuration
configure_environment_template(dynawo_sys_path, global_config_dir)

In [ ]:
# --- 3. IMPORT LIBRARIES AND BASE CONFIGURATION ---
import pandas as pd
import pypowsybl as pp
import pypowsybl.dynamic as dyn
import matplotlib.pyplot as plt
from IPython.display import display

# Ensure plots are displayed inline within the Jupyter environment
%matplotlib inline

print(f"PyPowSyBl version: {pp.__version__}")

current_working_dir = os.getcwd()
dynawo_project_dir = os.path.join(current_working_dir, "dynawo")
project_data_dir = os.path.join(current_working_dir, "getting_started_data")

## Step 1 & 2: Create Network and Initialize Loadflow Profiles

We construct the 2-bus network structure programmatically using the provided utility function.

In [ ]:
# --- 4. CREATE NETWORK AND INITIALIZE PROFILES ---
def build_tutorial_network() -> pp.network.Network:
    """
    Programmatically constructs a basic 2-bus static power network.
    """
    # Initialize an empty network using the correct PyPowSyBl function
    power_network = pp.network.load("getting_started_data/Base_Case.xiidm")

    # Create substations and voltage levels
    power_network.create_substations(
        id=["SUB_GEN", "SUB_LOAD"], name=["Generator Substation", "Load Substation"]
    )
    power_network.create_voltage_levels(
        id=["VL_GEN", "VL_LOAD"],
        name=["VL_GEN", "VL_LOAD"],
        substation_id=["SUB_GEN", "SUB_LOAD"],
        topology_kind=["BUS_BREAKER", "BUS_BREAKER"],
        nominal_v=[400.0, 400.0],
    )

    # Create buses
    power_network.create_buses(
        id=["BUS_GEN", "BUS_LOAD"],
        name=["BUS_GEN", "BUS_LOAD"],
        voltage_level_id=["VL_GEN", "VL_LOAD"],
    )

    # Add Generator (acting as Slack bus regulating voltage)
    power_network.create_generators(
        id="GEN_1",
        name="GEN_1",
        voltage_level_id="VL_GEN",
        bus_id="BUS_GEN",
        target_p=150.0,
        target_v=400.0,
        min_p=0.0,
        max_p=300.0,
        voltage_regulator_on=True,
    )

    # Add Load
    power_network.create_loads(
        id="LOAD_1",
        name="LOAD_1",
        voltage_level_id="VL_LOAD",
        bus_id="BUS_LOAD",
        p0=150.0,
        q0=50.0,
    )

    # Connect the buses with a transmission line
    power_network.create_lines(
        id="LINE_1",
        name="LINE_1",
        voltage_level1_id="VL_GEN",
        bus1_id="BUS_GEN",
        voltage_level2_id="VL_LOAD",
        bus2_id="BUS_LOAD",
        r=1.5,
        x=15.0,
        b1=0.0005,
        b2=0.0005,
    )

    return power_network


print("Creating power network structure...")
power_network = build_tutorial_network()

## Step 3: Run Loadflow

Execute the base Loadflow using PyPowSyBl's OpenLF. The Loadflow provider automatically identifies the Slack bus based on network topology.

In [ ]:
# --- 5. RUN STEADY-STATE LOADFLOW ---
def execute_and_display_loadflow(network: pp.network.Network):
    """Executes AC Load Flow analysis and displays the resulting bus metrics."""
    print("Running Loadflow (OpenLF)...")
    loadflow_parameters = pp.loadflow.Parameters()
    loadflow_results = pp.loadflow.run_ac(network, parameters=loadflow_parameters)

    print(f"Loadflow Status: {loadflow_results[0].status.name}")

    buses_dataframe = network.get_buses()
    display(buses_dataframe[["name", "v_mag", "v_angle"]])


# Execute loadflow
execute_and_display_loadflow(power_network)

## Step 4: Dynamic Models Setup

Map the static components to their dynamic representations. We map our static `GEN_1` to a precompiled **Modelica Synchronous Generator** model from the Dynawo library.

In [ ]:
# --- 6. CONFIGURE DYNAMIC MODELS MAPPING ---
def map_dynamic_models() -> dyn.ModelMapping:
    """Maps static network components to precompiled dynamic Modelica classes."""
    print("Mapping static elements to precompiled dynamic Modelica classes...")
    dynamic_mapping = dyn.ModelMapping()

    # Define mapping for the PV Generator
    generator_mapping_df = pd.DataFrame(
        [
            {
                "static_id": "GEN_1",
                "parameter_set_id": "GEN_1",
                "model_name": "GeneratorSynchronousThreeWindingsGoverNordicVRNordic",
            }
        ]
    ).set_index("static_id", drop=False)

    dynamic_mapping.add_synchronous_generator(generator_mapping_df)
    print("Dynamic mapping complete.")
    return dynamic_mapping


model_mapping = map_dynamic_models()

## Step 5: Physical Parameters and Loadflow Initialization

Construct the `.PAR` files using the strict physical properties required for `GeneratorPV`, the `LoadAlphaBeta` model, and the general Network. IIDM references (`<reference>`) retrieve $P$, $Q$, $V$, and $\theta$ from the Loadflow solution to guarantee seamless $t=0$ initialization.

In [ ]:
# --- 7. GENERATE PHYSICAL PARAMETERS (PAR FILES) ---
def generate_parameter_files():
    """Writes the XML parameter files for the dynamic models and static network."""
    print("Generating dynamic physical parameters and Loadflow initialization mapping...")

    xml_dynamic_model_parameters = """<?xml version="1.0" encoding="UTF-8"?>
<parametersSet xmlns="http://www.rte-france.com/dynawo">
    <set id="GEN_1">
        <par type="DOUBLE" name="generator_DPu" value="0"/>
        <par type="INT" name="generator_ExcitationPu" value="1"/>
        <par type="DOUBLE" name="generator_H" value="3"/>
        <par type="DOUBLE" name="generator_MdPuEfd" value="1"/>
        <par type="DOUBLE" name="generator_PNomAlt" value="4275"/>
        <par type="DOUBLE" name="generator_PNomTurb" value="4275"/>
        <par type="DOUBLE" name="generator_RTfPu" value="0"/>
        <par type="DOUBLE" name="generator_RaPu" value="0.002"/>
        <par type="DOUBLE" name="generator_SNom" value="4500"/>
        <par type="DOUBLE" name="generator_SnTfo" value="4500"/>
        <par type="DOUBLE" name="generator_Tpd0" value="5"/>
        <par type="DOUBLE" name="generator_Tppd0" value="0.05"/>
        <par type="DOUBLE" name="generator_Tppq0" value="0.1"/>
        <par type="DOUBLE" name="generator_UBaseHV" value="15"/>
        <par type="DOUBLE" name="generator_UBaseLV" value="15"/>
        <par type="DOUBLE" name="generator_UNom" value="15"/>
        <par type="DOUBLE" name="generator_UNomHV" value="15"/>
        <par type="DOUBLE" name="generator_UNomLV" value="15"/>
        <par type="DOUBLE" name="generator_XTfPu" value="0"/>
        <par type="DOUBLE" name="generator_XdPu" value="1.1"/>
        <par type="DOUBLE" name="generator_XlPu" value="0.15"/>
        <par type="DOUBLE" name="generator_XpdPu" value="0.25"/>
        <par type="DOUBLE" name="generator_XppdPu" value="0.2"/>
        <par type="DOUBLE" name="generator_XppqPu" value="0.2"/>
        <par type="DOUBLE" name="generator_XqPu" value="0.7"/>
        <par type="DOUBLE" name="generator_md" value="0.1"/>
        <par type="DOUBLE" name="generator_mq" value="0.1"/>
        <par type="DOUBLE" name="generator_nd" value="6.0257"/>
        <par type="DOUBLE" name="generator_nq" value="6.0257"/>
        <par type="DOUBLE" name="governor_KSigma" value="0.08"/>
        <par type="DOUBLE" name="governor_Ki" value="0.4"/>
        <par type="DOUBLE" name="governor_Kp" value="2"/>
        <par type="DOUBLE" name="governor_PNom" value="4275"/>
        <par type="DOUBLE" name="voltageRegulator_EfdMaxPu" value="4"/>
        <par type="DOUBLE" name="voltageRegulator_IrLimPu" value="1.8991"/>
        <par type="DOUBLE" name="voltageRegulator_KPss" value="0"/>
        <par type="DOUBLE" name="voltageRegulator_KTgr" value="70"/>
        <par type="DOUBLE" name="voltageRegulator_OelMode" value="0"/>
        <par type="DOUBLE" name="voltageRegulator_tDerOmega" value="1"/>
        <par type="DOUBLE" name="voltageRegulator_tLagPss" value="0.01"/>
        <par type="DOUBLE" name="voltageRegulator_tLagTgr" value="20"/>
        <par type="DOUBLE" name="voltageRegulator_tLeadPss" value="1"/>
        <par type="DOUBLE" name="voltageRegulator_tLeadTgr" value="10"/>
        <par type="DOUBLE" name="voltageRegulator_tOelMin" value="-11"/>
        <par type="BOOL" name="generator_UseApproximation" value="true"/>
        
        <reference type="DOUBLE" name="generator_P0Pu" origData="IIDM" origName="p_pu"/>
        <reference type="DOUBLE" name="generator_Q0Pu" origData="IIDM" origName="q_pu"/>
        <reference type="DOUBLE" name="generator_U0Pu" origData="IIDM" origName="v_pu"/>
        <reference type="DOUBLE" name="generator_UPhase0" origData="IIDM" origName="angle"/>
    </set>
</parametersSet>
"""

    xml_static_network_parameters = """<?xml version="1.0" encoding="UTF-8"?>
<parametersSet xmlns="http://www.rte-france.com/dynawo">
    <set id="Network">
        <par type="DOUBLE" name="capacitor_no_reclosing_delay" value="300.0"/>
        <par type="DOUBLE" name="dangling_line_currentLimit_maxTimeOperation" value="90.0"/>
        <par type="DOUBLE" name="line_currentLimit_maxTimeOperation" value="90.0"/>
        <par type="DOUBLE" name="load_Tp" value="90.0"/>
        <par type="DOUBLE" name="load_Tq" value="90.0"/>
        <par type="DOUBLE" name="load_alpha" value="1.0"/>
        <par type="DOUBLE" name="load_alphaLong" value="0.0"/>
        <par type="DOUBLE" name="load_beta" value="2.0"/>
        <par type="DOUBLE" name="load_betaLong" value="0.0"/>
        <par type="BOOL" name="load_isControllable" value="false"/>
        <par type="BOOL" name="load_isRestorative" value="false"/>
        <par type="DOUBLE" name="load_zPMax" value="100.0"/>
        <par type="DOUBLE" name="load_zQMax" value="100.0"/>
        <par type="DOUBLE" name="reactance_no_reclosing_delay" value="0.0"/>
        <par type="DOUBLE" name="transformer_currentLimit_maxTimeOperation" value="90.0"/>
        <par type="DOUBLE" name="transformer_t1st_HT" value="30.0"/>
        <par type="DOUBLE" name="transformer_t1st_THT" value="30.0"/>
        <par type="DOUBLE" name="transformer_tNext_HT" value="10.0"/>
        <par type="DOUBLE" name="transformer_tNext_THT" value="10.0"/>
        <par type="DOUBLE" name="transformer_tolV" value="0.014999999700000001"/>
        <par type="BOOL" name="BUS_LOAD_hasShortCircuitCapabilities" value="true"/>
    </set>
</parametersSet>
"""

    export_data_dir = "getting_started_data"
    os.makedirs(export_data_dir, exist_ok=True)

    with open(os.path.join(export_data_dir, "Base_Case.par"), "w") as file:
        file.write(xml_dynamic_model_parameters)

    with open(os.path.join(export_data_dir, "Network.par"), "w") as file:
        file.write(xml_static_network_parameters)

    print("Physical and network parameters successfully saved.")


# Execute file generation
generate_parameter_files()

## Step 6: Execute Dynamic Simulation

Introduce a temporal fault (short-circuit) at the load bus and track the terminal voltages of the system.

In [ ]:
# --- 8. EXECUTE DYNAMIC SIMULATION AND PLOT ---
def configure_and_run_simulation(network: pp.network.Network, mapping: dyn.ModelMapping):
    """Sets up the short-circuit event, runs the Dynawo simulation, and plots the results."""
    print("Configuring short-circuit event and observers...")

    # Map events (Fault at BUS_LOAD)
    event_mapping = dyn.EventMapping()
    event_mapping.add_node_fault(
        static_id="BUS_LOAD", start_time=2.0, fault_time=0.1, r_pu=0.0, x_pu=0.0001
    )

    # Map output curves
    output_mapping = dyn.OutputVariableMapping()
    output_mapping.add_standard_model_curves("BUS_GEN", "U_value")
    output_mapping.add_standard_model_curves("BUS_LOAD", "U_value")

    # Define simulation parameters
    sim_parameters = dyn.Parameters(
        start_time=0.0, stop_time=10.0, provider_parameters={"dynawoVersion": "1.8"}
    )

    dynawo_simulation = dyn.Simulation()

    print("Running dynamic time-domain simulation...")
    sim_results = dynawo_simulation.run(
        network,
        model_mapping=mapping,
        event_mapping=event_mapping,
        timeseries_mapping=output_mapping,
        parameters=sim_parameters,
    )

    if sim_results.status().name == "SUCCESS":
        print("Simulation completed successfully! Generating plots...")
        plot_simulation_results(sim_results.curves())
    else:
        print(f"CRITICAL ERROR: Simulation failed. Status: {sim_results.status_text()}")


def plot_simulation_results(curves_dataframe):
    """Plots the voltage transient responses."""
    plt.figure(figsize=(10, 5))

    plt.plot(
        curves_dataframe.index,
        curves_dataframe["NETWORK_BUS_LOAD_U_value"],
        label="Load Bus Voltage (PU)",
        color="red",
    )
    plt.plot(
        curves_dataframe.index,
        curves_dataframe["NETWORK_BUS_GEN_U_value"],
        label="Generator Bus Voltage (PU)",
        linestyle="--",
        color="blue",
    )

    plt.title("Transient Response: 2-Bus System Voltages")
    plt.xlabel("Time (Seconds)")
    plt.ylabel("Voltage (PU)")
    plt.grid(True)
    plt.legend()
    plt.show()


# Execute simulation
configure_and_run_simulation(power_network, model_mapping)